# Treatments repeated over time: which weekly offer policy brings the most full-price orders?

## At a glance

A subscription retailer sends each customer one offer a week for three weeks: nothing, a 15%
discount code, or a free-shipping code. Among active subscribers whose term runs through week 4,
which three-week plan, here called a policy, would leave the most placing a full-price order (one
without a code) in week 4 if all of them stayed tracked? The retailer's system drew each offer at
random, with logged probabilities that depended on the customer's engagement.

Compared with sending nothing, the win-back policy, a discount only to customers who are not
engaged, raises the share placing a full-price order by 0.075 (95% confidence interval 0.035 to
0.115), about 7 or 8 more such customers in 100. The interval covers sampling variation only and
assumes a correct model of who withdraws. A discount every week changes it by -0.088 (interval
-0.147 to -0.029), and win-back beats it by 0.163 (0.110 to 0.217). The data cannot separate
win-back from free shipping every week, or from win-back then ship (free shipping for engaged
customers instead of nothing), so no best policy among these three can be named.

Customers who withdraw tracking consent drop out of the data, so the answer assumes
that, given each customer's record, nothing unrecorded affects both who withdraws and who orders,
which the data cannot check.

The data are simulated; the true win-back effect is 0.072.

## The question

Before each weekly send, the retailer's system scores whether the customer is engaged and draws the
offer with probabilities that depend on that score. The retention lead wants to compare five
policies. Three send the same offer every week (`none`, `discount`, `shipping`). Win-back sends the
discount to a customer who is not engaged that week and nothing otherwise, and win-back then ship
sends free shipping instead of nothing. For each policy, the quantity we want is the share of
customers who would order at full price in week 4 if all of them had received that policy's offers
and stayed tracked, compared with sending nothing.

In the graph, an arrow means "causes", drawn for one week $t$ of weeks 1 to 3. $W$ is tenure (how
long the customer has subscribed), $L_t$ engagement, $A_t$ the offer, $C_t$ tracking consent after
the send, $D$ the number of discounts sent to engaged customers so far (built into the simulation,
not a column), $Y$ the week-4 order, and $U$ an unrecorded cause. Week-4 engagement, $L_4$, is not
recorded, which does no harm because it comes after the last offer.

```text
 W (tenure)  -->  L_1, L_2, L_3, L_4, C_1, C_2, C_3, and Y       (no arrow W --> A_t)

 One week t (t = 1, 2, 3):

   C_t  <--  L_t  -->  A_t  -->  L_{t+1}  --> ... -->  L_4  -->  Y
                |       |                                        ^
                |       v                                        |
                +-----> D  --------------------------------------+
   also: L_t --> L_{t+1}

 U (unrecorded cause)  - - ->  C_t and Y        (assumed absent)
```

Engagement is the trap. A confounder changes both the treatment and the outcome, and week-2
engagement is one for the week-2 offer, yet the week-1 offer also changes it. So one regression of
the order on the three offers fails: adjusting for week-2 engagement blocks part of the week-1
offer's effect, and leaving it out confounds the week-2 offer.

Engaged customers also keep tracking consent more often, so those still seen in week 4 are a
selected group. A discount sent to an engaged customer lowers the later order (through $D$), so a
rule that reads engagement can beat every fixed offer. The key assumption is that $U$ does not
exist: nothing unrecorded drives both consent loss and the order.

:::{dropdown} Details: what each arrow and missing arrow asserts

| arrow or absent arrow | what it asserts | role |
| --- | --- | --- |
| `W` to `L_1`, `L_2`, `L_3`, `L_4`, `C_t`, and `Y` | tenure shifts engagement, consent, and the order | baseline confounder |
| `L_t` to `A_t` | the logged probabilities read that week's engagement | confounder of the offer |
| `A_t` to `L_{t+1}` | a discount raises the next engagement, and free shipping raises it less | the treatment changes a later confounder |
| `L_t` to `L_{t+1}` | engagement persists | a path from a confounder to $Y$ |
| `L_t` to `C_t` | engaged customers keep tracking consent more often | censoring with a recorded cause |
| `L_t` and `A_t` to `D`, and `D` to `Y` | a discount sent to an engaged customer lowers the order | a direct offer path; engagement modifies the discount effect |
| `L_4` to `Y` | week-4 engagement raises the order | a mediator after the last send, which the data do not record |
| no unrecorded cause of `A_t` | the system randomizes | holds by design |
| no `U` with arrows into `C_t` and `Y` | consent loss depends only on the recorded history | an assumption that needs judgment; "What if customers who withdraw consent differ?" probes it |
| no `W` to `A_t` | the offers ignore tenure | a feature of this simulation; the analysis does not need it |
| no `A_t` to `C_t` | a send does not change that week's consent | a feature of this simulation; the analysis does not need it |

The graph has no competing event: the subscription term is fixed at time zero and runs through week
4, so no account closes before the outcome week. The quantity is $\psi_d = E[Y^{d,\bar c=1}]$, the
week-4 share under policy $d$ with tracking kept, and the contrasts are $\psi_d -
\psi_{\mathrm{none}}$. The sequential g-formula identifies it under sequential exchangeability and
positivity for the offers and for staying tracked, consistency, and no interference.
:::

## Load the tools

Every fit names its models and seeds, so a rerun prints the same numbers.

In [1]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.6


Another version of `cleverly` can print different numbers.

## The data

`make_sequential_offers` draws 20,000 simulated customers, one row per customer and week, and also
returns the true values, which real data never give you. The seed, 1000, was fixed before the page
first ran, so the draw was not picked for its results.

In [2]:
from cleverly.datasets import make_sequential_offers

# With your own data, load a long table here: one row per customer and week until the customer
# withdraws, with the offer, the covariates, a flag for still being tracked after the send, and
# the outcome repeated on each of the customer's rows.
# Real data have no `truth`, so drop every line that uses it.
panel, truth = make_sequential_offers(20_000, seed=1000)
print("rows and columns:", panel.shape)
print("customers:", panel["customer"].nunique())
print(list(panel.columns))
print(panel.head(6).round(3))
print()
left = (panel.groupby("customer")["consent"].min() == 0).sum()
print(f"customers who withdrew consent before week 4: {left} of {panel['customer'].nunique()}")
print("true values of the simulated data:")
for name, value in truth.items():
    print(f"  {name:36s} {value:.4f}")

rows and columns: (57154, 10)
customers: 20000
['customer', 'week', 'tenure', 'engaged', 'offer', 'consent', 'p_none', 'p_discount', 'p_shipping', 'full_price']
   customer  week  tenure  engaged     offer  consent  p_none  p_discount  p_shipping  full_price
0         0     1  -0.321        1      none        1     0.4        0.25        0.35         1.0
1         0     2  -0.321        0      none        1     0.3        0.45        0.25         1.0
2         0     3  -0.321        0  shipping        1     0.3        0.45        0.25         1.0
3         1     1  -0.486        1  shipping        1     0.4        0.25        0.35         NaN
4         1     2  -0.486        1      none        1     0.4        0.25        0.35         NaN
5         1     3  -0.486        0  shipping        0     0.3        0.45        0.25         NaN

customers who withdrew consent before week 4: 2686 of 20000
true values of the simulated data:
  ey_regimen[none]                     0.5275
  ey_regime

Each row is one customer in one week: `tenure` (standardized), that week's `engaged` score (1 or 0)
recorded before the send, the `offer` sent, and `consent`, 1 while the customer still allows
tracking. `p_none`, `p_discount`, and `p_shipping` are the logged offer probabilities. A customer's
rows stop after they withdraw, and `full_price`, the week-4 order, repeats on every row and is
missing for the 2686 customers who withdrew. Keep them: the fit uses their early weeks, and
dropping them would leave only the selected group who stayed tracked.

In the true values, `ey_regimen` is the expected week-4 share under a policy and `ate_regimen` its
average effect against no offers.

## Why a simple comparison misleads

Three arrows of the graph show up directly in the data. A quick analysis would then compare, among
customers still tracked after week 3, those whose offers happened to match a policy with those who
happened to get no offer in any week.

In [3]:
from cleverly.datasets import OFFER_PLANS

# This cell only shows why a simple comparison misleads, so you can skip it on your own data.
# To run it there, change the column names and the week numbers (1, 2, 3; 3 is the last offer
# week).

week_one = panel[panel["week"] == 1]
week_two = panel[panel["week"] == 2][["customer", "engaged"]]
moved = week_one.merge(week_two, on="customer", suffixes=("", "_week2"))
print("share engaged in week 2, by the week-1 offer:")
print(moved.groupby("offer")["engaged_week2"].mean().round(3).to_string())
print()
print("share of each week-1 offer, by week-1 engagement (1 = engaged):")
print(pd.crosstab(week_one["engaged"], week_one["offer"], normalize="index").round(3).to_string())
print()
print("share still tracked after the week-1 send, by week-1 engagement:")
print(week_one.groupby("engaged")["consent"].mean().round(3).to_string())
print()

# For your own data, OFFER_PLANS becomes a dict from each policy name to a function that takes
# an array of 0/1 engagement scores and returns one offer per customer (a fixed policy
# returns the same offer for everyone).
wide = panel.pivot(index="customer", columns="week")
kept = wide[wide[("consent", 3)] == 1]
first_engaged = kept[("engaged", 1)].astype(int).to_numpy()
rows = []
for name, policy_rule in OFFER_PLANS.items():
    # A customer follows a policy when each week's offer is the policy's offer for that
    # week's engagement.
    followed = np.ones(len(kept), dtype=bool)
    for week in (1, 2, 3):
        engaged = kept[("engaged", week)].astype(int).to_numpy()
        followed &= kept[("offer", week)].to_numpy() == np.asarray(
            policy_rule(engaged), dtype=object
        )
    rows.append(
        {
            "plan": name,
            "followers": int(followed.sum()),
            "share": kept[("full_price", 1)][followed].mean(),
            "engaged_week1": first_engaged[followed].mean(),
        }
    )
crude = pd.DataFrame(rows).set_index("plan")
crude["crude_vs_none"] = crude["share"] - crude.loc["none", "share"]
crude["true_vs_none"] = [truth.get(f"ate_regimen[{name} vs none]", 0.0) for name in crude.index]
print(f"customers still tracked after week 3: {len(kept)} of {len(wide)}")
print(f"share engaged in week 1 among these customers: {first_engaged.mean():.3f}")
print(crude.round(3).to_string())

share engaged in week 2, by the week-1 offer:
offer
discount    0.886
none        0.551
shipping    0.661

share of each week-1 offer, by week-1 engagement (1 = engaged):
offer    discount   none  shipping
engaged                           
0           0.440  0.308     0.251
1           0.246  0.399     0.355

share still tracked after the week-1 send, by week-1 engagement:
engaged
0    0.929
1    0.969

customers still tracked after week 3: 17314 of 20000
share engaged in week 1 among these customers: 0.562
               followers  share  engaged_week1  crude_vs_none  true_vs_none
plan                                                                       
none                 805  0.555          0.646          0.000         0.000
discount             436  0.445          0.404         -0.110        -0.068
shipping             550  0.609          0.671          0.054         0.059
win_back            1247  0.604          0.524          0.049         0.072
win_back_ship        956  0.60

All three arrows show up: a week-1 discount raises week-2 engagement (0.886 engaged, against 0.551
after no offer), engaged customers more often get no offer (0.399 against 0.308), and they stay
tracked more often (0.969 against 0.929).

In the last table, the simple comparison (`crude_vs_none`) puts win-back 0.049 above no offers,
against a true 0.072 (`true_vs_none`), and puts both engagement-based policies too low. Customers
who happened to get no offer in all three weeks were more often engaged in week 1 (0.646, against
0.562 among all tracked customers), and engaged customers order more, which flatters the no-offer
group. A fair comparison accounts for engagement week by week, and for who stays tracked.

:::{dropdown} Details: free shipping, and the simple comparison over 100 draws
Free shipping also goes to engaged customers more often (0.355 against 0.251), so its followers are
about as engaged as the no-offer group (0.671 against 0.646), and its simple comparison, 0.054,
lands near the true 0.059.

A simulation we ran for this page (script
`reviews/notebook-review/probes/sequential-offers/sweep.py`) repeats the page's fits on 100 draws,
seeds 1000 to 1099; the later Details blocks quote it too. There the simple win-back comparison
fell below the truth on 96 of 100 draws, by 0.0314 on average. The simulation's own standard error
(Monte Carlo standard error) was 0.0019. The simple shipping comparison had a mean bias of -0.0061
(Monte Carlo standard error 0.0029), about two standard errors, so the simulation does not show
that it is zero.
:::

## Write down the study design

The study description (`StudyProtocol`) says who is in the study, when follow-up starts, the
policies, the outcome, and how dropouts count. It does not change the estimate; it records what the
estimate means.

In [4]:
from cleverly.datasets import offers_protocol

# For your own study, fill in the same ten fields, in the order the output prints them:
#   target_population, eligibility: who the answer is about, and the entry rules;
#   time_zero, horizon: when follow-up starts, and when the outcome is measured;
#   treatment_strategies, treatment_versions: the policies compared, and exactly what each sends;
#   outcome: the measured result;
#   intercurrent_event_handling: how events such as consent withdrawal count;
#   interference_unit: the unit whose offers can affect only its own outcome;
#   assumption_rationale: why the design makes the causal assumptions believable.
# protocol = cleverly.StudyProtocol(
#     target_population=..., eligibility=(...), time_zero=..., treatment_strategies=(...),
#     treatment_versions=(...), outcome=..., horizon=..., intercurrent_event_handling=(...),
#     interference_unit=..., assumption_rationale=(...),
# )
protocol = offers_protocol()
print("\n".join(protocol.summary_lines()))

Population: Active subscribers at the start of the campaign window
Eligibility: Active subscription at the first weekly send; Marketing consent at the first weekly send; At least one prior order; One account per customer, so shared-household accounts are excluded; A subscription term that is fixed at the first weekly send and runs through week 4
Follow-up starts: The first weekly send, after the week-1 engagement score and before the first offer
Strategies: Send no offer in weeks 1 to 3; Send the 15% discount code every week; Send the free-shipping code every week; Win-back: send the discount code when the customer is not engaged, otherwise send no offer; Win-back then ship: send the discount code when the customer is not engaged, otherwise send the free-shipping code
Versions: No code sent; The declared 15% code, one send per week, valid until the end of that week; The declared free-shipping code, one send per week, valid until the end of that week; The declared 15% code for customers

Follow-up starts before the first offer, so no offer changes who is in the study. Codes expire at
the end of their week, so no week-4 order uses a code, and the term runs through week 4, so no
account closes early. The output says withdrawal "censors" the customer, meaning their later data
are missing; under the "hypothetical" strategy, the question is the order they would have placed
had tracking continued. Marketing consent, which lets the retailer send offers, is an entry rule.

## Columns, policies, and what must hold

`LongPanel` names the columns: customer, week, offer, `tenure` (measured once), `engaged` (changes
each week), the consent flag, and the outcome. `treatment_probabilities` points each offer to its
logged-probability column, so the fit uses the known probabilities instead of a model of who got
which offer. A policy is a tuple with one offer per week, or a rule (`DynamicRegimen`) that reads
the customer's engagement that week; `regime` and `regimen` are the library's words for a policy.
`study.identify(...)` then lists what must be true for the answer to be causal.

In [5]:
from cleverly import CausalStudy, LongPanel, RegimeContrast
from cleverly.datasets import OFFER_WEEKS, win_back, win_back_then_ship
from cleverly.longitudinal import DynamicRegimen, each_node

# For your own data, map each offer to its logged-probability column. Without logged
# probabilities, leave out treatment_probabilities and pass a treatment_learner to ModelSpec.
# The offers' exchangeability then becomes an assumption, as it is for consent, and the
# estimate needs either the outcome models or both the treatment and consent models to be right.
LOGGED = {"none": "p_none", "discount": "p_discount", "shipping": "p_shipping"}


def offers_design(*, time_varying=("engaged",)):
    # For your own data, change the column names.
    return LongPanel(
        unit="customer",
        time="week",
        treatment="offer",
        baseline=["tenure"],
        time_varying=list(time_varying),
        censoring="consent",
        outcome="full_price",
        treatment_probabilities=LOGGED,
    )


def rule(label, offer_for):
    # offer_for takes an array of 0/1 engagement scores and returns one offer name per customer;
    # win_back returns "discount" where the score is 0 and "none" where it is 1. The rule reads
    # the customer's record at each week, where column x is named "x@<week>". rule_kind="known"
    # says the rule was fixed in advance, not estimated from these data.
    return DynamicRegimen(
        label,
        each_node(lambda history, week: offer_for(history[f"engaged@{week}"]), OFFER_WEEKS),
        rule_kind="known",
    )


# For your own data, OFFER_WEEKS is your number of decision weeks. Each entry of plans is one
# policy: a tuple with one offer per week, or rule(name, f) with your own function f.
study = CausalStudy(panel, design=offers_design(), protocol=protocol)
plans = {
    "none": ("none",) * OFFER_WEEKS,
    "discount": ("discount",) * OFFER_WEEKS,
    "shipping": ("shipping",) * OFFER_WEEKS,
    "win_back": rule("win_back", win_back),
    "win_back_ship": rule("win_back_ship", win_back_then_ship),
}
effect = study.identify(RegimeContrast(plans, reference="none"))
print(effect.summary())

                         Causal question (ate_regime)
Question:   contrast of each regime     Rows:                           20,000
            against the reference       Answered by:     adjusting for history
Outcome:    full_price (binary)
Treatment:  offer@1, offer@2, offer@3
Baseline:   tenure
What each treatment time adjusts for
treatment   adjusted for
------------------------------------------------------------------------------
offer@1     tenure, engaged@1
offer@2     tenure, engaged@1, engaged@2, offer@1
offer@3     tenure, engaged@1, engaged@2, engaged@3, offer@1, offer@2
Conditions the answer rests on
condition                            checkable   what it says
------------------------------------------------------------------------------
well-defined treatment               no          Each unit's observed outcomes
                                                 are the ones it would have
                                                 had under the treatments it
    

The summary lists what the analysis adjusts for before each week's offer (a node is one
week's send): tenure, the engagement scores so far, and the earlier offers. Check that every
variable you mean to adjust for appears there. The effect comes from adjusting for them
(`explicit-adjustment`) week by week, working back from the last week (the sequential g-formula),
and fits the helper pieces: an outcome model per week, the offer
probabilities (here read from the logged columns), and the consent model.

Well-defined treatment, also called consistency, means each customer's recorded engagement and order
are what they would show under the offers actually sent, which needs each offer to be one declared
code. No interference means one customer's offers do not change another's order. No hidden
confounding at each time means that each week, given the record so far, nothing unrecorded drove
both the offer, or the consent loss, and the order; randomization covers the offers, but the
consent half needs judgment. Overlap at each time means every customer had a real chance of the policy's offer
and of staying tracked each week: the logged probabilities are all at least 0.25, and "Check the
weights" reads the consent part.

## Estimate each policy's effect

The method, longitudinal targeted maximum likelihood estimation (LTMLE), is built for treatments
that repeat over time. It works backward: a model predicts the week-4 order from the week-3 record
with the week-3 offer set to the policy's, that prediction becomes the target of a model at week 2,
and so on to week 1. At each week, a correction step that weights each customer by one over their
chance of following the policy and staying tracked removes the bias the outcome model leaves.
Because the offer probabilities are known, the estimate stays on target in large samples if either
the outcome models or the consent model is right. Here the outcome models are simple and miss part
of the truth, so the interval rests on the consent model: when it is right, the interval holds, if
anything slightly wide.

In [6]:
from cleverly import CrossFitting, ModelSpec, Runtime, TMLEMethod

# For your own data, swap in your learners. With flexible learners such as gradient boosting,
# use CrossFitting(n_folds=5), so each row's predictions come from models trained without it.
logistic = LogisticRegression(max_iter=1000, random_state=42)
ltmle = TMLEMethod(
    models=ModelSpec(
        outcome_learner=logistic,  # the week-4 order (0 or 1) from the week-3 record
        pseudo_learner=LinearRegression(),  # earlier weeks, whose targets are predictions
        censoring_learner=logistic,  # the chance of staying tracked each week
    ),
    cross_fitting=CrossFitting(enabled=False),  # fit these simple models on all rows
    runtime=Runtime(random_state=42, n_jobs=1),  # fix the seed
)
result = effect.estimate(method=ltmle)
print(result.summary())


def against_truth(fit):
    rows = []
    for key, estimate in fit.estimates.items():
        low, high = estimate.ci
        rows.append(
            {
                "contrast": key.removeprefix("ate_regimen[").removesuffix(" vs none]"),
                "estimate": estimate.psi,
                "std_error": estimate.std_error,
                "ci_low": low,
                "ci_high": high,
                "truth": truth[key],
                "covers": bool(low <= truth[key] <= high),
            }
        )
    return pd.DataFrame(rows).set_index("contrast")


print()
print(against_truth(result).round(3).to_string())

                        Longitudinal estimate (LTMLE)
Question:      full_price under each    Units:                          20,000
               plan; each row is        Time points:                         3
               defined below            Out-of-fold predictions:            no
Outcome:       full_price (binary), at  Outcome model:    logistic regression;
               the end                                   linear regression for
Treatment:     offer@1, offer@2,                           earlier time points
               offer@3                  Treatment model:      known (declared)
Adjusted for:  tenure; later:           Censoring model:   logistic regression
               engaged@1, engaged@2,    Probability limits:          [0.01, 1]
               engaged@3
Censoring:     consent@1, consent@2,
               consent@3; each plan
               keeps every unit
               observed
Method:        longitudinal targeted
               maximum likelihood
          

Each row of the top table is one policy against no offers: win-back gains 0.075 (95% interval 0.035
to 0.115), and the bottom table, from `against_truth`, shows every interval holding its true value.
`experimental` in the `support` column means no validation study covers this exact setup: the
nearest one differs in the treatment (here three offer levels) and in the logged
probabilities. Mention the label when you report the numbers.

The plan table counts each policy's followers and their weights, which the weight check below
reads; `rule d` is the policy's rule at each week. The header's probability limits put a floor
of 0.01 under each chance of following a plan, which caps a weight at 100, and note 3 says that
floor moved 0.2% of the customers still on free shipping at week 3. The simultaneous bands are
wider intervals that hold for all four policies together, so quote them when you claim several
policies beat no offers together. Free shipping's interval only just excludes zero (from 0.001),
and its band, -0.012 to 0.125, includes zero, so its gain is the least certain.

:::{dropdown} Details: the outcome models, and 100 repeated draws
The outcome models' main terms omit the discount-fatigue interaction of the offer and engagement,
and the true regression also averages over the unrecorded week-4 engagement, which interacts with
tenure. The consent model, a logistic regression on the record, matches how the simulation draws
consent loss. With a wrong outcome model and a correctly specified parametric consent model, the
usual interval from the fit is conservative (wider than it needs to be) ([known node
mechanisms](../technical-reference/longitudinal-tmle.md#known-node-mechanisms)). In the simulation,
the four intervals covered on 95, 95, 92, and 94 of 100 draws.
:::

## Comparing two policies on the same customers

To ask whether win-back beats another policy, subtract their two effects. Both effects come from
the same customers and share the same no-offer group, so their errors move together, as in a paired
t-test. `result.contrast(...)` computes the standard error of the difference with that pairing. The
unpaired formula, `sqrt(se_a^2 + se_b^2)`, treats the two effects as independent.

In [7]:
pairs = [
    ("win_back", "discount"),
    ("win_back", "shipping"),
    ("win_back_ship", "win_back"),
]
rows = []
for better, other in pairs:
    keys = [f"ate_regimen[{better} vs none]", f"ate_regimen[{other} vs none]"]
    paired = result.contrast(
        lambda values: values[0] - values[1], keys, name=f"{better} minus {other}"
    )
    unpaired_se = float(np.hypot(result[keys[0]].std_error, result[keys[1]].std_error))
    low, high = paired.ci
    rows.append(
        {
            "comparison": f"{better} - {other}",
            "estimate": paired.psi,
            "paired_se": paired.std_error,
            "paired_low": low,
            "paired_high": high,
            "unpaired_se": unpaired_se,
            "truth": truth[keys[0]] - truth[keys[1]],
        }
    )
comparison = pd.DataFrame(rows).set_index("comparison")
print(comparison.round(3).to_string())

                          estimate  paired_se  paired_low  paired_high  unpaired_se  truth
comparison                                                                                
win_back - discount          0.163      0.027       0.110        0.217        0.036  0.140
win_back - shipping          0.018      0.026      -0.032        0.069        0.035  0.013
win_back_ship - win_back     0.007      0.020      -0.033        0.046        0.031  0.027


Win-back beats a weekly discount by 0.163, with a paired interval of 0.110 to 0.217 (the true gap
is 0.140). Win-back against free shipping (0.018, interval -0.032 to 0.069) and win-back then ship
against win-back (0.007, interval -0.033 to 0.046) both include zero, so the data cannot rank these
policies. The unpaired standard error is larger in every row, 0.036 against 0.027 in the first,
because the shared noise partly cancels when you subtract. The unpaired interval is too wide, so
report the paired one.

:::{dropdown} Details: the comparisons over 100 draws
In the simulation, the paired standard error was below the unpaired one on every draw for all three
comparisons. The paired intervals of the last two comparisons excluded zero on 8 and 26 of 100
draws, so at this sample size the data seldom separate those policies.
:::

## Check the weights and the consent model

`assess()` checks that the correction step converged, how heavy each policy's weights are, and how
well the consent model predicts. A customer's weight is one over their chance of following the
policy and staying tracked, so a heavy weight means few similar customers back the estimate. The
effective sample size is how many equally weighted customers would carry the same information, and
the fit caps every weight at 100.

In [8]:
assessment = result.assess()
attention = tuple(item.name for item in assessment.attention)
print("checks that need attention:", attention or "none")
print()
support = assessment.report("support").to_frame()
# 3 is the last offer week; use your own.
last_week = support[support["time"] == 3].set_index("regimen")
labels = {
    "n_followed": "followers",
    "max_weight": "largest weight",
    "effective_n": "effective sample size",
    "share_truncated": "share at the cap",
}
print("customers who followed each policy in all three weeks:")
shown = last_week[list(labels)].rename(columns=labels)
shown["effective sample size"] = shown["effective sample size"].round().astype(int)
print(shown.round({"largest weight": 1, "share at the cap": 3}))
smallest_share = (support["effective_n"] / support["n_followed"]).min()
print(f"smallest effective sample size, as a share of followers, in any week: {smallest_share:.1%}")
print()
nuisance = assessment.report("nuisance_models").to_frame()
consent = nuisance[nuisance["role"] == "censoring"]
print("consent model, measured on the rows it was fitted on:")
for _, row in consent.iterrows():
    print(
        f"  week {row['time']}: AUC {row['auc']:.3f},",
        f"calibration slope {row['calibration_slope']:.3f}",
    )

checks that need attention: none

customers who followed each policy in all three weeks:
               followers  largest weight  effective sample size  share at the cap
regimen                                                                          
none                 805            59.7                    709             0.000
discount             436            80.9                    379             0.000
shipping             550           100.0                    475             0.002
win_back            1247            19.7                   1242             0.000
win_back_ship        956            28.6                    936             0.000
smallest effective sample size, as a share of followers, in any week: 86.4%

consent model, measured on the rows it was fitted on:
  week 1: AUC 0.647, calibration slope 1.003
  week 2: AUC 0.645, calibration slope 1.004
  week 3: AUC 0.650, calibration slope 1.000


Nothing needs attention. Win-back keeps the most followers (1247) and the lightest weights (largest
19.7), because it sends the likeliest offer in each engagement group, while a fixed offer is a less
likely one for one group: the weekly discount keeps 436. Free shipping reaches the cap for one of
its 550 followers, a never-engaged customer with low tenure and so a small chance of staying
tracked. Every effective sample size is at least 86.4% of its follower count, so no small group
dominates.

The consent model separates customers only modestly (AUC about 0.65), because withdrawal varies
little with engagement and tenure, and its calibration slopes sit at 1, the ideal (its predicted
chances are neither too spread out nor too bunched). Both are measured on the rows it was fitted
on, so they cannot show it is right. Warning signs on your own data are few followers, many
customers at the cap, or effective sample sizes far below the follower counts.

:::{dropdown} Details: why the capped weight comes from the consent model
The offer probabilities alone cannot reach the cap of 100, because the product of the three
smallest logged probabilities is 0.0156. The capped customer's three chances of staying tracked
multiply to less than 0.64, which takes the cumulative probability below 0.01.
:::

## Try to break the result

These checks change the data in a known way and refit the win-back effect 20 times each. The
placebo swaps in random offers that follow the week-to-week pattern of the logged ones but ignore
the customer, so the estimate should be about zero. The random common cause adds a column of pure
noise at every week (`noise_nodes=True`), so the estimate should barely move. The subset check
refits on random 70% samples, whose estimates should spread as much as the standard error predicts
for that size, and the bootstrap check refits on customers drawn with replacement. None of them can
detect a hidden cause of withdrawal.

In [9]:
from cleverly.validation import refute

refutation = refute(
    result,
    estimand="ate_regimen[win_back vs none]",
    tests=("placebo", "subset", "random_common_cause", "bootstrap_stability"),
    noise_nodes=True,  # add the noise column at every week
    n_replicates=20,
)
columns = ["test", "original", "refuted_mean", "refuted_sd", "expectation", "verdict"]
print(refutation.to_frame()[columns].round(4).to_string(index=False))
print()
for test in refutation.tests:
    if test.statistic_name:
        print(f"{test.name}: {test.statistic_name} = {test.statistic:.4f}")

               test  original  refuted_mean  refuted_sd                            expectation     verdict
            placebo    0.0754       -0.0024      0.0181                                    ~ 0 descriptive
             subset    0.0754        0.0720      0.0125                      scatter ~ 0.01339 descriptive
random_common_cause    0.0754        0.0751      0.0013                              ~ 0.07536 descriptive
bootstrap_stability    0.0754        0.0753      0.0160 spread 0.01597 around the original fit descriptive

placebo: t = -0.5872
subset: scatter ratio rho = 0.9313
random_common_cause: paired shift in standard errors = -0.0125


The placebo refits average -0.0024, close to zero, as they should be; their `t`, the average over
its own standard error, is -0.59, well inside 2. The 70% refits spread about as much as predicted
(`rho` 0.93, near the ideal 1), so the standard error of 0.020 is about the right size. The noise
column barely moves the estimate, and the bootstrap refits spread by about four fifths of the
standard error, which 20 refits cannot tell apart from it, so neither looks like a warning.
`descriptive` in the `verdict` column means no simulation study backs a pass or fail for these
checks on this kind of fit, so you judge these numbers yourself.

:::{dropdown} Details: what backs these checks, and checks that cannot run
No study sets the placebo threshold of 2 for this fit, and none says how much noise movement or
bootstrap spread is too much; for LTMLE, the refutation study covers only time-to-event outcomes.

A negative-control outcome needs a measure that the offers cannot affect and that the fit does not
adjust for. The data record only `tenure`, which the fit adjusts for, and `refute` refuses a
control that equals an adjustment column. The future-treatment check, which tests whether a later
offer appears to change a measurement taken before it, needs a binary treatment, and these offers
have three levels. The generated-outcome checks replace the outcome with a Gaussian draw, so they
refuse a 0-or-1 outcome, and the plasmode check, which refits on data simulated to mimic these,
also needs binary treatments without logged probabilities.
:::

## What if customers who withdraw consent differ?

The offers were randomized, so the assumption at risk is the one about who withdraws. `cleverly`
has no bound on a hidden cause of withdrawal for treatments over time, so the code drops `engaged`,
a real cause of withdrawal here, and refits the fixed-offer policies; the rules need `engaged`, and
`tenure` is the only other covariate. The logged offer probabilities stay, so any move comes from
the outcome and consent models.

In [10]:
static = {name: plans[name] for name in ("none", "discount", "shipping")}
full_fit = study.identify(RegimeContrast(static, reference="none")).estimate(method=ltmle)
# For your own data, drop each recorded cause of consent loss in turn.
reduced_study = CausalStudy(panel, design=offers_design(time_varying=()), protocol=protocol)
reduced_fit = reduced_study.identify(RegimeContrast(static, reference="none")).estimate(
    method=ltmle
)
rows = []
for key, estimate in full_fit.estimates.items():
    move = reduced_fit[key].psi - estimate.psi
    rows.append(
        {
            "contrast": key.removeprefix("ate_regimen[").removesuffix(" vs none]"),
            "with_engagement": estimate.psi,
            "without": reduced_fit[key].psi,
            "move": move,
            "move_in_se": move / estimate.std_error,
        }
    )
benchmark = pd.DataFrame(rows).set_index("contrast")
print(benchmark.round(3).to_string())

          with_engagement  without   move  move_in_se
contrast                                             
discount           -0.088   -0.095 -0.007      -0.239
shipping            0.057    0.054 -0.003      -0.095


Dropping `engaged`, a real cause of withdrawal, moves the discount effect from -0.088 to -0.095 and
the shipping effect from 0.057 to 0.054, a quarter of a standard error or less (`move_in_se`). This
shows that the fixed-offer effects lean little on engagement once tenure is kept. It says nothing
about how far a hidden cause would move them, since one that matters more, or acts through other
paths, could move them further; someone who knows why customers withdraw must judge whether such a
cause exists. The rules, win-back among them, need engagement, so this check gives them no number.

:::{dropdown} Details: the sensitivity tools that do not apply here
The E-value, a measure of how strong a hidden cause would have to be to explain the effect away,
needs a model of hidden confounding week by week for this kind of quantity, and `cleverly` derives
none. The simulated hidden-confounder replay holds one fixed hidden variable on data without
dropout, and this panel has consent censoring and three offer levels.
:::

## What we can and can't conclude

Among active subscribers whose term runs through week 4, had every customer stayed tracked,
win-back would raise the week-4 full-price share by about 0.075 over sending nothing (95% interval
0.035 to 0.115), and a weekly discount would lower it. Win-back beats the weekly discount, but the
data cannot rank it against free shipping or win-back then ship. The intervals cover sampling
variation only. They assume that, given each customer's record, nothing unrecorded affects both who
withdraws and who orders, and that the consent model is right. They also assume that each offer
means its declared code and that one customer's offers do not change another's. The data cannot
check the hidden-cause, declared-code, or no-spillover assumptions, and the consent checks above
cannot confirm the consent model.

This is one simulated sample with a known answer, so it says nothing about real customers or about
how the method performs in general.

## Go deeper

### How we know the method works

A simulation study, [LTMLE with known treatment
probabilities](../technical-reference/method-evidence/known-node-mechanisms-ltmle.md), checks this
method's bias and interval coverage on other simulated data, including a case like this page's, with
the treatment probabilities known, the censoring model fitted, and a wrong outcome model.
[Week-by-week g-computation and
weighting](../technical-reference/method-evidence/sequential-baselines.md) checks the simpler
methods, which the [sequential baselines entry](../technical-reference/sequential-baselines.md)
defines. No study covers three offer levels over three weeks. The [technical
entry](../technical-reference/longitudinal-tmle.md#validation-issues-special-to-this-method) lists
the method's evidence and limits.

### When the outcome models are useless

Because the offer probabilities are known, LTMLE needs only one of the other two pieces to be
right: the outcome models or the consent model. G-computation, which averages the outcome models'
predictions with no correction step, needs the outcome models. To see the difference, the code
refits both with a deliberately useless outcome model, which predicts the average order of the
policy's followers and ignores their record. The consent model stays the logistic regression of the
main fit. With these models g-computation gives a point estimate only, because its interval needs
`GLM` (generalized linear model) outcome models.

In [11]:
from cleverly import GComputationMethod

# This check needs the true effects, so it runs only on simulated data.
blind = ModelSpec(
    outcome_learner=DummyClassifier(strategy="prior"),
    pseudo_learner=DummyRegressor(),
    censoring_learner=logistic,
)
blind_ltmle = effect.estimate(
    method=TMLEMethod(models=blind, cross_fitting=ltmle.cross_fitting, runtime=ltmle.runtime)
)
blind_gcomp = effect.estimate(method=GComputationMethod(models=blind, runtime=ltmle.runtime))
print(
    "same estimands:",
    blind_ltmle.estimates.keys() == blind_gcomp.estimates.keys() == result.estimates.keys(),
)
rows = []
for key, estimate in blind_ltmle.estimates.items():
    low, high = estimate.ci
    name = key.removeprefix("ate_regimen[").removesuffix(" vs none]")
    rows.append(
        {
            "contrast": name,
            "truth": truth[key],
            "ltmle": result[key].psi,
            "ltmle_se": result[key].std_error,
            "blind_ltmle": estimate.psi,
            "blind_se": estimate.std_error,
            "ci_low": low,
            "ci_high": high,
            "covers": bool(low <= truth[key] <= high),
            "blind_gcomp": blind_gcomp[key].psi,
            "crude": crude.loc[name, "crude_vs_none"],
        }
    )
blind_table = pd.DataFrame(rows).set_index("contrast")
print(blind_table.round(3).to_string())

same estimands: True
               truth  ltmle  ltmle_se  blind_ltmle  blind_se  ci_low  ci_high  covers  blind_gcomp  crude
contrast                                                                                                 
discount      -0.068 -0.088     0.030       -0.096     0.033  -0.160   -0.032    True       -0.110 -0.110
shipping       0.059  0.057     0.028        0.068     0.030   0.009    0.126    True        0.054  0.054
win_back       0.072  0.075     0.020        0.080     0.022   0.038    0.123    True        0.049  0.049
win_back_ship  0.099  0.082     0.024        0.088     0.025   0.038    0.138    True        0.050  0.050


G-computation with the useless model (`blind_gcomp`) equals the simple comparison (`crude`) on
every row, because that model returns the followers' average, so for win-back it gives 0.049
against a true 0.072. LTMLE with the same model (`blind_ltmle`) gives 0.080 for win-back, with an
interval of 0.038 to 0.123 that holds the truth, and all four of its intervals hold their true
values. The correction step uses the known probabilities and the consent model to repair the
useless prediction, at the price of a wider standard error, 0.022 against 0.020.

:::{dropdown} Details: the three estimators, and 100 repeated draws
G-computation is consistent when every outcome regression is correct. Inverse probability
weighting, which this page does not fit, reads the offer probabilities and the consent model, and
is consistent when the consent model is correct. LTMLE reads all of them, and is consistent when
every outcome regression is correct or the consent model is correct, because the offer
probabilities are exact; both hold up to the cap on the weights. The test
`tests/unit/test_known_node_mechanisms.py` declares every factor known and shows that LTMLE then
returns the truth under a wrong outcome regression. The case with the consent model fitted rests on
the theory in [known node
mechanisms](../technical-reference/longitudinal-tmle.md#known-node-mechanisms) and on the
simulation study above.

In the simulation, the wrong-model g-computation estimate for win-back fell below the truth by
0.0314 on average. The wrong-model LTMLE bias was -0.0060, and its interval covered the truth on 93
of 100 draws.
:::

### A dry run on fresh simulated customers

A real campaign would try the chosen policy on a pilot group and compare its order rate with the
estimate. Here `policy="win_back"` simulates a pilot of 10,000 new customers who all receive
win-back and stay tracked. The code estimates the win-back share itself (`RegimeMean`) and compares
the two; the samples are independent, so the difference has standard error `sqrt(se_fit^2 +
se_pilot^2)`.

In [12]:
from cleverly import RegimeMean

# A real pilot would be a table of customers who were sent win-back.
pilot, _ = make_sequential_offers(10_000, seed=101_000, policy="win_back")
# The week-4 order repeats on every row, so the week-3 rows give one order per customer.
pilot_outcome = pilot.loc[pilot["week"] == 3, "full_price"]
levels = study.identify(RegimeMean(plans)).estimate(method=ltmle)
fitted = levels["ey_regimen[win_back]"]
pilot_mean = float(pilot_outcome.mean())
pilot_se = float(pilot_outcome.std() / np.sqrt(len(pilot_outcome)))
difference = fitted.psi - pilot_mean
difference_se = float(np.hypot(fitted.std_error, pilot_se))
print(
    f"pilot customers: {len(pilot_outcome)}; lost to consent: {int((pilot['consent'] == 0).sum())}"
)
print(f"pilot share of full-price orders:  {pilot_mean:.3f}  (standard error {pilot_se:.4f})")
print(
    f"estimated win-back share:          {fitted.psi:.3f}  (standard error {fitted.std_error:.4f})"
)
print(f"difference, estimate minus pilot:  {difference:+.4f}  (standard error {difference_se:.4f})")
print(f"difference in standard errors:     {difference / difference_se:+.2f}")
print(f"true win-back share:               {truth['ey_regimen[win_back]']:.3f}")

pilot customers: 10000; lost to consent: 0
pilot share of full-price orders:  0.609  (standard error 0.0049)
estimated win-back share:          0.596  (standard error 0.0133)
difference, estimate minus pilot:  -0.0135  (standard error 0.0141)
difference in standard errors:     -0.95
true win-back share:               0.600


The pilot's share is 0.609 and the estimate is 0.596, a difference of -0.0135, or -0.95 standard
errors, well inside the usual 1.96. Both samples come from simulated data in which every assumption
holds, so this tests the method only. On real data, a pilot that disagreed could point to a broken
assumption, and one that agreed would not prove the assumptions.

## Where to go next

The [sequential pricing example](sequential-pricing.ipynb) studies a price changed each month in
the same retailer program. The [longitudinal user guide](../user-guide/longitudinal.md) shows how
to read a long table of repeated measurements and how to report the simpler week-by-week methods.
The [examples index](index.md) lists every example.